# Contract

A contract is the base concept from which you can calculate energy costs.
It consists of 4 components:
- a `provider` tariff
- a `distributor` tariff
- `fees` — which are essentially government tariffs — multiple fee tariffs can be provided as a list and will be merged automatically
- a `tax_rate` (a percentage applied to all tariff costs)

Given a contract and consumption data, you can calculate the cost of that consumption under the terms of the contract.

The output DataFrame uses a structured MultiIndex with fixed, translatable labels based on `TariffCategory` and `CostGroup` enums, making it predictable and easy to use for i18n.

In [1]:
from zoneinfo import ZoneInfo

import pandas as pd

from energy_cost import Contract, Meter, Tariff
from energy_cost.data import CustomerType
from energy_cost.data.be.flanders.electricity import data

contract = Contract(
    provider=Tariff.from_yaml("../examples/tariffs/fixed.yml"),
    distributor=data.distributors["fluvius_imewo"],
    fees=data.fees[CustomerType.RESIDENTIAL],
    taxes=data.taxes,
    timezone=ZoneInfo("Europe/Brussels"),
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption])

timestamp    provider        distributor              \
                            consumption  total    capacity consumption   
                                  total  total       total      levies   
0 2024-01-01 00:00:00+01:00       59.52  59.52    8.209764    0.799532   
1 2024-02-01 00:00:00+01:00       55.68  55.68    8.209764    0.747949   
2 2024-03-01 00:00:00+01:00        0.02   0.02    8.209764    0.000269   

                                                                        \
                                                       fixed             
  public_service_fee      total transmission data_management     total   
0          16.610008  23.676520     6.266980        1.209508  1.209508   
1          15.538395  22.149003     5.862659        1.131475  1.131475   
2           0.005581   0.007956     0.002106        1.207883  1.207883   

                            fees                                        total  \
       total         consumption                            total       total   
       total energy_contribution     excise      total      total       total   
0  33.095793            1.146415  28.260096  29.406511  29.406511  129.343642   
1  31.490243            1.072452  26.436864  27.509316  27.509316  121.560333   
2   9.425603            0.000385   0.009496   0.009881   0.009881   10.022813   

      taxes  
      total  
      total  
0  7.321338  
1  6.880774  
2  0.567329

> **Note**: most tariffs are based on indexes, make sure to define all required indexes before calculating costs.
>
> See the [index documentation](./index.ipynb) for more details.
>
> If you are defining your own tariffs, also have a look at the [tariff documentation](./tariff.ipynb) for details on how to implement tariffs.
>
> The [tax documentation](./tax.ipynb) can also be usefull if you want to define your own taxes.

### Resolution

By default the cost is calculated for each month, but you can specify a different resolution if you want to calculate costs for different time periods eg yearly.

In [2]:
import isodate

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption], resolution=isodate.Duration(years=1))

timestamp    provider                          distributor  \
                            consumption     fixed          total    capacity   
                                  total fixed_fee  total   total       total   
0 2024-01-01 00:00:00+01:00      702.72       0.0    0.0  702.72   98.517173   
1 2025-01-01 00:00:00+01:00      630.72     120.0  120.0  750.72  133.105596   
2 2026-01-01 00:00:00+01:00      135.96     180.0  180.0  315.96   33.875614   

                                                                           \
  consumption                                                       fixed   
       levies public_service_fee       total transmission data_management   
0    9.439638         196.105260  279.535692    73.990794           14.28   
1   12.631219         227.689219  412.792925   172.472486           17.51   
2    1.626422          28.234133   59.240491    29.379936           17.85   

                                    fees                                      \
               total         consumption                               total   
   total       total energy_contribution      excise       total       total   
0  14.28  392.332864           13.535090  333.651456  347.186546  347.186546   
1  17.51  563.408521           13.498109  332.739840  346.237949  346.237949   
2  17.85  110.966105            2.182271   53.794840   55.977111   55.977111   

         total      taxes  
         total      total  
         total      total  
0  1528.773775  86.534365  
1  1759.988458  99.621988  
2   511.877409  28.974193

### Time range

By default start and end are set to the first and last timestamp in the data, but you can specify a different start and end if you want to calculate costs for a different time period than the one covered by the data.

This is useful for the Flemish capacity tariff that uses a rolling average of the consumption of the last 12 months to determine the cost for the next month. In this case you would need to provide data for at least 12 months before the start of the period you want to calculate costs for.

In [3]:
import datetime as dt
from zoneinfo import ZoneInfo

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": [0.002] * 4 * 24 * 365 + [0.003] * 4 * 24 * 365 + [0.003] * 4 * 24 * 60 + [0.003],
        }
    )
)

contract.calculate_cost(
    [consumption],
    start=dt.datetime(2025, 1, 1, tzinfo=ZoneInfo("Europe/Brussels")),
    end=dt.datetime(2025, 12, 31, tzinfo=ZoneInfo("Europe/Brussels")),
)

timestamp    provider                         distributor  \
                             consumption     fixed         total    capacity   
                                   total fixed_fee total   total       total   
0  2025-01-01 00:00:00+01:00      803.52      10.0  10.0  813.52   38.452728   
1  2025-02-01 00:00:00+01:00      725.76      10.0  10.0  735.76   39.931679   
2  2025-03-01 00:00:00+01:00      802.44      10.0  10.0  812.44   41.410630   
3  2025-04-01 00:00:00+02:00      777.60      10.0  10.0  787.60   42.889581   
4  2025-05-01 00:00:00+02:00      803.52      10.0  10.0  813.52   44.368532   
5  2025-06-01 00:00:00+02:00      777.60      10.0  10.0  787.60   45.847483   
6  2025-07-01 00:00:00+02:00      803.52      10.0  10.0  813.52   47.326434   
7  2025-08-01 00:00:00+02:00      803.52      10.0  10.0  813.52   48.805385   
8  2025-09-01 00:00:00+02:00      777.60      10.0  10.0  787.60   50.284336   
9  2025-10-01 00:00:00+02:00      804.60      10.0  10.0  814.60   51.763287   
10 2025-11-01 00:00:00+01:00      777.60      10.0  10.0  787.60   53.242238   
11 2025-12-01 00:00:00+01:00      803.52      10.0  10.0  813.52   53.242238   

                                                                            \
   consumption                                                       fixed   
        levies public_service_fee       total transmission data_management   
0    16.091827         290.069827  525.886877   219.725222        1.487151   
1    14.534554         261.998554  474.994598   198.461491        1.343233   
2    16.070198         289.679948  525.180040   219.429893        1.485152   
3    15.572736         280.712736  508.922784   212.637312        1.439178   
4    16.091827         290.069827  525.886877   219.725222        1.487151   
5    15.572736         280.712736  508.922784   212.637312        1.439178   
6    16.091827         290.069827  525.886877   219.725222        1.487151   
7    16.091827         290.069827  525.886877   219.725222        1.487151   
8    15.572736         280.712736  508.922784   212.637312        1.439178   
9    16.113456         290.459706  526.593714   220.020552        1.489150   
10   15.572736         280.712736  508.922784   212.637312        1.439178   
11   16.091827         290.069827  525.886877   219.725222        1.487151   

                                        fees                          \
                   total         consumption                           
       total       total energy_contribution      excise       total   
0   1.487151  565.826755           17.196221  406.114744  423.310965   
1   1.343233  516.269510           15.532070  366.813317  382.345388   
2   1.485152  568.075821           17.173108  405.568891  422.741999   
3   1.439178  553.251543           16.641504  393.014268  409.655772   
4   1.487151  571.742559           17.196221  406.114744  423.310965   
5   1.439178  556.209445           16.641504  393.014268  409.655772   
6   1.487151  574.700462           17.196221  406.114744  423.310965   
7   1.487151  576.179413           17.196221  406.114744  423.310965   
8   1.439178  560.646298           16.641504  393.014268  409.655772   
9   1.489150  579.846151           17.219334  406.660597  423.879931   
10  1.439178  563.604200           16.641504  393.014268  409.655772   
11  1.487151  580.616266           17.196221  406.114744  423.310965   

                      total       taxes  
         total        total       total  
         total        total       total  
0   423.310965  1910.817183  108.159463  
1   382.345388  1732.437392   98.062494  
2   422.741999  1911.453289  108.195469  
3   409.655772  1855.537754  105.030439  
4   423.310965  1917.087936  108.514411  
5   409.655772  1858.673131  105.207913  
6   423.310965  1920.223312  108.691886  
7   423.310965  1921.791000  108.780623  
8   409.655772  1863.376195  105.474124  
9   423.879931  1927.425647  109.099565  
10  409.655772  1866

As you can see in the above example, while the capacity was constant in 2025, the first months are still cheaper then the last, as the cost of the capacity tariff is based on the consumption of the previous 12 months, which includes the cheaper months of 2024.

### Injection

Aside from a consumption timeseries, you can also provide injection as a separate timeseries, as these are often measured independently and have different tariffs. If you provide injection data, the cost of the injection will be calculated separately and subtracted from the cost of the consumption, as injection is essentially negative consumption.

In [4]:
from energy_cost import PowerDirection

injection = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    ),
    direction=PowerDirection.INJECTION,
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2026-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption, injection], resolution=isodate.Duration(years=1))

timestamp    provider                                      \
                            consumption     fixed        injection    total   
                                  total fixed_fee  total     total    total   
0 2024-01-01 00:00:00+01:00      702.72       0.0    0.0  -140.544  562.176   
1 2025-01-01 00:00:00+01:00      630.72     120.0  120.0  -105.120  645.600   
2 2026-01-01 00:00:00+01:00      135.96     180.0  180.0   -28.325  287.635   

  distributor                                                          \
     capacity consumption                                               
        total      levies public_service_fee       total transmission   
0   98.517173    9.439638         196.105260  279.535692    73.990794   
1  133.105596   12.631219         227.689219  412.792925   172.472486   
2   33.875614    1.626422          28.234133   59.240491    29.379936   

                                                    fees              \
            fixed              total         consumption               
  data_management  total       total energy_contribution      excise   
0           14.28  14.28  392.332864           13.535090  333.651456   
1           17.51  17.51  563.408521           13.498109  332.739840   
2           17.85  17.85  110.966105            2.182271   53.794840   

                                 total      taxes  
                    total        total      total  
        total       total        total      total  
0  347.186546  347.186546  1388.229775  86.534365  
1  346.237949  346.237949  1654.868458  99.621988  
2   55.977111   55.977111   483.552409  28.974193

## Gas contracts

A contract can also be used to calculate gas costs, in which case the provider and distributor tariffs will be based on the consumed volume of gas instead of the power consumption. The rest of the concepts remain the same, but the tariffs will be different as they are based on different units.

In [5]:
from energy_cost.data.be.flanders.gas import data

contract = Contract(
    provider=Tariff.from_yaml("../examples/tariffs/fixed.yml"),
    distributor=data.distributors["fluvius_imewo"],
    fees=data.fees[CustomerType.RESIDENTIAL],
    taxes=data.taxes,
    timezone=ZoneInfo("Europe/Brussels"),
)

consumption = Meter(
    data=pd.DataFrame(
        {
            "timestamp": pd.date_range("2024-01-01T00:00:00+01:00", "2024-03-01T00:00:00+01:00", freq="15min"),
            "value": 0.0002,
        }
    )
)

contract.calculate_cost([consumption])

timestamp    provider                   distributor  \
                            consumption  total            consumption   
                                  total  total fixed_distribution_fee   
0 2024-01-01 00:00:00+01:00       59.52  59.52               7.620410   
1 2024-02-01 00:00:00+01:00       55.68  55.68               7.128770   
2 2024-03-01 00:00:00+01:00        0.02   0.02               0.002561   

                                                           \
                                                            
  other_levies pension_levy proportional_distribution_fee   
0     0.064341     0.071960                      4.562029   
1     0.060190     0.067317                      4.267705   
2     0.000022     0.000024                      0.001533   

                                                                             \
                                                 fixed                total   
  public_service_obligation      total data_management     total      total   
0                  0.282006  12.600746        1.114645  1.114645  13.715391   
1                  0.263812  11.787794        1.042732  1.042732  12.830527   
2                  0.000095   0.004234        1.113147  1.113147   1.117381   

                 fees                                        total     taxes  
          consumption                            total       total     total  
  energy_contribution     excise      total      total       total     total  
0            1.146415  28.260096  29.406511  29.406511  108.800415  6.158514  
1            1.072452  26.436864  27.509316  27.509316  101.781034  5.761191  
2            0.000385   0.009496   0.009881   0.009881    1.216098  0.068836